In [1]:
# ==========================================
# 套件安裝與導入區域
# ==========================================
import subprocess
import sys

# 確保必要的高階套件已安裝
for pkg in ['statsmodels', 'plotly', 'ipywidgets', 'kaleido', 'scikit-learn', 'yfinance', 'gymnasium', 'stable_baselines3']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import warnings
warnings.filterwarnings('ignore')

import os
import time
import math
import sqlite3
import logging
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from joblib import Parallel, delayed

pd.set_option('display.float_format', '{:.4f}'.format)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
# ==========================================
# 參數區域 (Global Parameters)
# ==========================================
FAST_TEST_MODE = True

if FAST_TEST_MODE:
    print("啟動【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。")
    START_DATE = '2019-01-01'
    END_DATE = '2022-12-31'
    # TARGET_SECTOR = 'Information Technology'
    TARGET_SECTOR = None
else:
    print("啟動【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。")
    START_DATE = '2000-01-01'
    END_DATE = '2025-12-31'
    TARGET_SECTOR = None

# 資料庫與快取配置
DB_PATH = r'..\data\sp500.db'
USE_DYNAMIC_SECTORS = True
IMPUTED_SECTOR_PATH = r'..\data\imputed_sectors.csv'

# 視窗滾動參數
FORMATION_WINDOW = 252       # 形成期(前一年)
TRADING_WINDOW = 126         # 交易期(約半年)
ROLLING_WINDOW = 21          # 滾動步長(約一個月)
MIN_HISTORY_DAYS = 200

# 配對與統計指定參數
MAX_PAIRS_PER_TRANCHE = 5    # 決定每個梯隊要執行的最大配對數
COINT_P_VALUE = 0.01         # 嚴格的共整合 p-value 門檻

# ==========================================
# 特徵工程與過濾參數 (Feature Engineering Parameters)
# ==========================================
Z_WINDOW = 63                # Z-Score 與共變異數的滾動視窗 (天)
RSI_WINDOW = 14              # 價差 RSI 的計算週期 (天)
PCA_COMPONENTS = 3           # PCA 提取的市場主成分數量
MAX_HURST = 0.45             # 赫斯特指數 (Hurst) 上限 (確保均值回歸)
MAX_HALF_LIFE = 126          # 半衰期上限 (天) (確保在交易期內能收斂)

# ==========================================
# 交易執行與資金控管參數 (Trading & Risk Parameters)
# ==========================================
TRANSACTION_COST = 0.0029    # 雙邊交易手續費(0.29%)
STOP_LOSS_PCT = -0.30        # 單筆配對交易的硬性停損線 (-30%)
MAX_HOLD_DAYS = 63           # 最大持倉天數
INITIAL_CAPITAL = 10000    # 初始本金
CAPITAL_TRANCHES = math.ceil(TRADING_WINDOW / ROLLING_WINDOW) + 1# 浮動視窗資金切割份數
TRANCHE_ALLOCATION = INITIAL_CAPITAL / CAPITAL_TRANCHES

# ==========================================
# 強化學習 (RL) 訓練超參數 (Global RL Parameters)
# ==========================================
RL_LEARNING_RATE = 0.0005    # 學習率 (建議範圍: 1e-4 ~ 5e-4，太高容易崩潰，太低學太慢)
RL_GAMMA = 0.99              # 折扣因子 (越接近1，Agent越看重長期回報而非短期獲利)
RL_ENT_COEF = 0.005           # 熵係數 (增加探索)
RL_BATCH_SIZE = 128           # 批次大小 (神經網路每次更新的樣本數)
RL_TOTAL_TIMESTEPS = 50000   # 總訓練步數 (降低以避免 OOS 過擬合)

os.makedirs(os.path.dirname(IMPUTED_SECTOR_PATH), exist_ok=True)

啟動【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。


In [3]:
# ==========================================
# 模組 1: 數據管理與基礎預處理 (Data Pipeline)
# ==========================================
class FetchDataRL:
    def __init__(self, db_path, start_date, end_date, target_sector, use_dynamic, min_days, imputed_path):
        self.db_path = db_path
        self.start_date = start_date
        self.end_date = end_date
        self.target_sector = target_sector
        self.use_dynamic = use_dynamic
        self.imputed_path = imputed_path
        self.min_days = min_days

    def _fix_unknown_sectors(self, sector_df, prices_df):
        if not self.use_dynamic:
            print("不使用動態產業補齊，維持原始 Unknown 分類作為對照組。")
            return sector_df
        
        if os.path.exists(self.imputed_path):
            print(f"從本機快取載入已補齊的產業分類: {self.imputed_path}")
            cached_df = pd.read_csv(self.imputed_path)
            update_df = cached_df.set_index('ticker')
            sector_df = sector_df.set_index('ticker')
            sector_df.update(update_df)
            return sector_df.reset_index()

        unknown_mask = sector_df['sector'] == 'Unknown'
        unknown_tickers = sector_df[unknown_mask]['ticker'].tolist()

        if not unknown_tickers:
            return sector_df

        print(f"找不到本機快取，正在透過 API 補齊 {len(unknown_tickers)} 檔 Unknown 股票的產業分類...")
        yf_logger = logging.getLogger('yfinance')
        original_level = yf_logger.level
        yf_logger.setLevel(logging.CRITICAL)

        fixed_sectors = []
        for i, ticker in enumerate(unknown_tickers):
            try:
                info = yf.Ticker(ticker).info
                sector = info.get('sector', 'Unknown')
                fixed_sectors.append({'ticker': ticker, 'sector': sector})
                time.sleep(0.02)
            except Exception:
                fixed_sectors.append({'ticker': ticker, 'sector': 'Unknown'})
            
            if (i + 1) % 50 == 0:
                print(f"已處理 {i + 1}/{len(unknown_tickers)}...")

        yf_logger.setLevel(original_level)
        fetched_df = pd.DataFrame(fixed_sectors)
        fetched_df.to_csv(self.imputed_path, index=False)
        print(f"API抓取完畢！已將動態產業分類永久儲存至: {self.imputed_path}")

        update_df = fetched_df.set_index('ticker')
        sector_df = sector_df.set_index('ticker')
        sector_df.update(update_df)
        sector_df = sector_df.reset_index()
        
        remaining = len(sector_df[sector_df['sector'] == 'Unknown'])
        print(f"補齊完成！剩餘真實無法識別(已下市)的 Unknown 股票數量: {remaining}")
        return sector_df

    def fetch_and_preprocess(self):
        print("從資料庫載入原始資料...")
        abs_db_path = os.path.abspath(self.db_path)
        print(f"目前嘗試連接的資料庫絕對路徑為: {abs_db_path}")

        prices_raw = None
        sector_info = None

        if not os.path.exists(self.db_path):
            print(f"【警告】找不到資料庫檔案，請確認路徑正確性！")
        else:
            try:
                conn = sqlite3.connect(self.db_path)
                
                price_queries = [
                    f"SELECT date, ticker, adj_close AS close FROM daily_prices WHERE date BETWEEN '{self.start_date}' AND '{self.end_date}'",
                    f"SELECT date, ticker, close FROM stock_prices WHERE date BETWEEN '{self.start_date}' AND '{self.end_date}'",
                    f"SELECT date, ticker, close FROM daily_prices WHERE date BETWEEN '{self.start_date}' AND '{self.end_date}'"
                ]
                for q in price_queries:
                    try:
                        prices_raw = pd.read_sql_query(q, conn, parse_dates=['date'])
                        if len(prices_raw) > 0:
                            print(f"成功加載價格數據: {len(prices_raw):,} 筆")
                            break
                    except Exception:
                        continue

                sector_queries = [
                    "SELECT ticker, sector FROM tickers",
                    "SELECT ticker, sector FROM sp500_components GROUP BY ticker"
                ]
                for q in sector_queries:
                    try:
                        sector_info = pd.read_sql_query(q, conn)
                        if len(sector_info) > 0:
                            print(f"成功加載行業數據: {len(sector_info):,} 筆")
                            break
                    except Exception:
                        continue
                
                if prices_raw is None or sector_info is None:
                    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
                    print(f"【除錯資訊】目前資料庫內擁有的資料表: {tables['name'].tolist()}")

                conn.close()

                if prices_raw is None or len(prices_raw) == 0:
                    raise RuntimeError("成功連接資料庫，但找不到對應的價格資料表或無符合時間範圍之數據。")
                if sector_info is None or len(sector_info) == 0:
                    raise RuntimeError("成功連接資料庫，但找不到對應的產業分類資料表。")

            except Exception as e:
                print(f"【資料庫讀取失敗】: {e}")
                prices_raw = None

        if prices_raw is None or sector_info is None:
            print("將生成測試用模擬數據 (為確保程式可獨立運行)...")
            dates = pd.date_range(self.start_date, self.end_date, freq='B')
            tickers = [f"TECH{i}" for i in range(1, 72)]
            prices_raw = pd.DataFrame({
                'date': np.tile(dates, len(tickers)),
                'ticker': np.repeat(tickers, len(dates)),
                'close': np.random.normal(100, 10, len(dates) * len(tickers))
            })
            sector_info = pd.DataFrame({'ticker': tickers, 'sector': ['Information Technology']*len(tickers)})

        sector_info = self._fix_unknown_sectors(sector_info, prices_raw)

        if self.target_sector is not None:
            target_tickers = sector_info[sector_info['sector'] == self.target_sector]['ticker'].tolist()
            print(f"板塊過濾: 篩選 {self.target_sector} 板塊之股票，共 {len(target_tickers)} 檔")
            prices_raw = prices_raw[prices_raw['ticker'].isin(target_tickers)]
            sector_info = sector_info[sector_info['sector'] == self.target_sector]
        else:
            print("全市場模式: 使用全部 S&P 500 股票")

        pivot = prices_raw.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
        pivot.index = pd.to_datetime(pivot.index)
        pivot.sort_index(inplace=True)
        pivot.ffill(limit=5, inplace=True)
        valid = pivot.columns[pivot.notna().sum() >= self.min_days]
        pivot = pivot[valid]
        print(f'Matrix: {len(pivot)} days x {len(pivot.columns)} tickers')

        print("Fetching VIX data...")
        try:
            vix_data = yf.download("^VIX", start=pivot.index.min(), end=pivot.index.max() + pd.Timedelta(days=1), progress=False)
            vix_close = vix_data['Close'].squeeze() if isinstance(vix_data.columns, pd.MultiIndex) else vix_data['Close']
            vix_df = pd.DataFrame({'VIX': vix_close})
            vix_df.index = pd.to_datetime(vix_df.index).tz_localize(None)
            vix_shifted = vix_df.shift(1)
            vix_aligned = vix_shifted.reindex(pivot.index).ffill()
            print("VIX feature matrix aligned.")
        except Exception as e:
            print(f"Error fetching VIX: {e}, using dummy VIX")
            vix_aligned = pd.DataFrame(20 + np.random.normal(0, 2, len(pivot)), index=pivot.index, columns=['VIX'])

        sector_map = sector_info.set_index('ticker')['sector'].to_dict()
        print(f"\n最終股票池統計:\n  股票數量: {len(pivot.columns)}\n  時間範圍: {self.start_date} 至 {self.end_date}")
        
        return prices_raw, pivot, vix_aligned, sector_map

In [4]:
# ==========================================
# 模組 2 & 3: 特徵工程、HDBSCAN 分群與共整合檢定
# ==========================================
class PairSelector:
    @staticmethod
    def compute_hurst(ts):
        if len(ts) < 20: return 0.5
        lags = range(2, 20)
        with np.errstate(invalid='ignore', divide='ignore'):
            var_diff = [np.var(ts[lag:] - ts[:-lag]) for lag in lags]
            valid = [v > 0 and not np.isnan(v) for v in var_diff]
            if sum(valid) < 5: return 0.5
            poly = np.polyfit(np.log(np.array(lags)[valid]), np.log(np.array(var_diff)[valid]), 1)
            return poly[0] / 2.0

    @staticmethod
    def compute_half_life(ts):
        z_lag = np.roll(ts, 1)
        z_lag[0] = 0
        z_ret = ts - z_lag
        z_ret[0] = 0
        z_lag2 = sm.add_constant(z_lag)
        try:
            res = sm.OLS(z_ret[1:], z_lag2[1:]).fit()
            hl = -np.log(2) / res.params[1]
            return hl if (hl > 0 and hl < 100) else 15.0
        except:
            return 15.0

    @staticmethod
    def extract_micro_ts_features(ts):
        ticker = ts.name
        try:
            ts_vals = ts.dropna()
            if len(ts_vals) < 30:
                return tuple([ticker] + [np.nan] * 7)
            
            log_ret = np.log(ts_vals / ts_vals.shift(1)).dropna()
            log_ret_latest = log_ret.iloc[-1]
            atr_latest = ts_vals.rolling(14).std().iloc[-1]
            
            roll_mean = ts_vals.rolling(60).mean()
            roll_std = ts_vals.rolling(60).std()
            z_score_latest = ((ts_vals - roll_mean) / roll_std).iloc[-1]
            
            vol_imb_latest = 0.0
            hurst = PairSelector.compute_hurst(ts_vals.values)
            half_life = PairSelector.compute_half_life(ts_vals.values)
            
            try:
                adf_res = adfuller(ts_vals.values, maxlag=1, regression='c', autolag=None)
                adf_stat = adf_res[0]
            except:
                adf_stat = np.nan
                
            return ticker, log_ret_latest, atr_latest, z_score_latest, vol_imb_latest, hurst, half_life, adf_stat
        except Exception:
            return tuple([ticker] + [np.nan] * 7)

    @staticmethod
    def feature_engineering_pipeline(price_pivot, vix_series, n_jobs=-1, n_pca_components=PCA_COMPONENTS, scaler_in=None):
        print("啟動特徵工程 Pipeline...")
        log_returns = np.log(price_pivot / price_pivot.shift(1)).fillna(0)
        
        pca = PCA(n_components=n_pca_components)
        market_factors = pca.fit_transform(log_returns)
        reconstructed = pca.inverse_transform(market_factors)
        pca_residuals = log_returns - reconstructed
        pca_res_latest = pca_residuals.iloc[-1]
        
        mf_1_series = pd.Series(market_factors[:, 0], index=log_returns.index)
        market_corr = log_returns.apply(lambda x: x.iloc[-20:].corr(mf_1_series.iloc[-20:]))

        tickers = price_pivot.columns
        print(f"啟動多執行緒處理 {len(tickers)} 檔股票之時間序列與 ADF 微觀特徵...")
        
        results = Parallel(n_jobs=n_jobs)(
            delayed(PairSelector.extract_micro_ts_features)(price_pivot[t]) for t in tickers
        )
        
        cols = ['ticker', 'Log_Ret', 'ATR', 'Z_Score', 'Vol_Imbalance', 'Hurst', 'Half_life', 'ADF_stat']
        features_df = pd.DataFrame([r for r in results if len(r) == 8], columns=cols).set_index('ticker')
        
        features_df['PCA_Res'] = pca_res_latest
        features_df['Market_Corr'] = market_corr
        features_df['VIX'] = vix_series.iloc[-1] if not vix_series.empty else np.nan
        features_df = features_df.dropna()

        mask = (features_df['Hurst'] <= MAX_HURST) & (features_df['Half_life'] < MAX_HALF_LIFE)
        features_df = features_df[mask]

        if scaler_in is None:
            scaler = StandardScaler()
            fit_mode = True
        else:
            scaler = scaler_in
            fit_mode = False

        feature_columns = ['Log_Ret', 'ATR', 'Z_Score', 'Vol_Imbalance', 'Hurst', 'Half_life', 'ADF_stat', 'PCA_Res', 'Market_Corr', 'VIX']
        valid_cols = [c for c in feature_columns if c in features_df.columns]

        if features_df.empty:
            print(f"▲警告: 經過 Hurst <= {MAX_HURST} 與 Half_life < {MAX_HALF_LIFE} 篩選後，剩餘 0 檔股票，跳過此梯隊。")
            if fit_mode: scaler.fit(np.zeros((1, len(valid_cols))))
            return pd.DataFrame(columns=valid_cols), scaler

        if fit_mode:
            std_matrix = scaler.fit_transform(features_df[valid_cols])
        else:
            std_matrix = scaler.transform(features_df[valid_cols])
            
        final_standardized_df = pd.DataFrame(std_matrix, index=features_df.index, columns=valid_cols)
        print(f"特徵矩陣建置完成！維度: {final_standardized_df.shape}")
        return final_standardized_df, scaler

    @staticmethod
    def test_single_coint(a, b, sector, price_window, p_threshold, min_len):
        series_a = price_window.get(a)
        series_b = price_window.get(b)
        if series_a is None or series_b is None: return None
        
        aligned = pd.concat([series_a, series_b], axis=1, join='inner').dropna()
        if len(aligned) < min_len: return None
        y, x = aligned.iloc[:, 0], aligned.iloc[:, 1]
        
        try:
            x_const = sm.add_constant(x)
            res = sm.OLS(y, x_const).fit()
            beta = res.params.iloc[1]
            score, pvalue, _ = coint(y, x, maxlag=1)
            
            if pvalue <= p_threshold and 0.5 < beta <= 2.0:
                return {'stock_a': a, 'stock_b': b, 'sector': sector, 'p_value': pvalue, 'beta': beta}
        except:
            pass
        return None

    @staticmethod
    def select_pairs_with_hdbscan(price_window, features_df, sector_map, top_ns, coint_pval=COINT_P_VALUE):
        if isinstance(top_ns, int): top_ns = [top_ns]
        
        features_df = features_df.copy()
        features_df['sector'] = features_df.index.map(lambda x: sector_map.get(x, 'Unknown'))
        feature_cols = [c for c in features_df.columns if c != 'sector']
        cluster_labels = pd.Series(index=features_df.index, dtype=int)
        cluster_labels[:] = -1
        global_offset = 0
        
        for sector, group in features_df.groupby('sector'):
            if len(group) < 2: continue
            X = group[feature_cols].values
            clusterer = HDBSCAN(min_cluster_size=2, metric='euclidean', cluster_selection_method='eom')
            labels = clusterer.fit_predict(X)
            
            new_labels = []
            for l in labels:
                if l == -1: new_labels.append(-1)
                else: new_labels.append(l + global_offset)
            cluster_labels.loc[group.index] = new_labels
            if len(set(labels)) > 1:
                global_offset += max(labels) + 1

        unique_clusters = set(cluster_labels) - {-1}
        candidates = []
        for cid in unique_clusters:
            cluster_tickers = cluster_labels[cluster_labels == cid].index.tolist()
            if len(cluster_tickers) >= 2:
                s_map = features_df.loc[cluster_tickers[0], 'sector']
                for a, b in combinations(cluster_tickers, 2):
                    candidates.append((a, b, s_map))

        if not candidates: return {n: [] for n in top_ns}
        print(f"完成排列組合，共產生 {len(candidates)} 組配對")

        min_len = len(price_window) * 0.8
        norm = price_window / price_window.iloc[0]
        
        coint_results = Parallel(n_jobs=-1)(
            delayed(PairSelector.test_single_coint)(a, b, sector, price_window, coint_pval, min_len) 
            for a, b, sector in candidates
        )
        passed = [res for res in coint_results if res is not None]
        if not passed: return {n: [] for n in top_ns}
        print(f"P-value 檢定過濾後，剩餘 {len(passed)} 組配對。")

        valid_pairs = []
        for p in passed:
            try:
                a_norm = norm[p['stock_a']]
                b_norm = norm[p['stock_b']]
                spread = a_norm - p['beta'] * b_norm
                spread_std = spread.std()
                
                if spread_std < 0.0020: continue
                
                centered_spread = spread - spread.mean()
                zero_crossings = ((centered_spread.shift(1) * centered_spread) < 0).sum()
                if zero_crossings < 4: continue
                
                p['spread_std'] = spread_std
                p['ssd'] = (spread**2).sum()
                p['zero_cross'] = zero_crossings
                p['trade_count'] = max(zero_crossings, 1)
                p['profit_space'] = spread.max() - spread.min()
                p['profit_per_trade'] = p['profit_space'] / p['trade_count']
                valid_pairs.append(p)
            except:
                continue
                
        df_passed = pd.DataFrame(valid_pairs)
        if df_passed.empty: 
            print("  -> 但經過「獲利空間(>0.58%)」與「過零率(>=12次)」嚴格校驗後，全數遭到淘汰。")
            return {n: [] for n in top_ns}

        print(f"  -> 經過獲利空間與過零率校驗後，最終剩餘 {len(df_passed)} 組配對可以進行評分。")

        df_passed['ssd_rank'] = df_passed['ssd'].rank(ascending=True)
        df_passed['profit_rank'] = df_passed['profit_per_trade'].rank(ascending=False)
        df_passed['final_score'] = df_passed['ssd_rank'] + df_passed['profit_rank']
        df_passed = df_passed.sort_values('final_score')
        
        print("SSD 排序完成，前部配對:")
        for i, (_, row) in enumerate(df_passed.head(5).iterrows()):
            print(f"  {i+1}: {row['stock_a']} vs {row['stock_b']} (SSD: {row['ssd']:.4f}, Beta: {row['beta']:.4f})")
            
        return {n: df_passed.head(n).to_dict('records') for n in top_ns}

In [5]:
# ==========================================
# 模組 4: 強化學習 MDP 環境 (Gymnasium)
# ==========================================
class PairsTradingEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self, data_df, transaction_cost=TRANSACTION_COST, stop_loss_pct=STOP_LOSS_PCT, max_hold_days=MAX_HOLD_DAYS, beta=1.0, ablation_vix=False):
        super(PairsTradingEnv, self).__init__()
        self.df = data_df.reset_index(drop=True)
        self.max_steps = len(self.df) - 1
        self.tc = transaction_cost
        self.stop_loss_pct = stop_loss_pct
        self.max_hold_days = max_hold_days
        self.beta = beta
        self.ablation_vix = ablation_vix
        
        self.action_space = spaces.Discrete(3) # 0: Flat, 1: Long, 2: Short
        self.obs_dim = 8 # 新增 RSI 捕捉動能轉折
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32)
        
        self.current_step = 0
        self.current_pos = 0
        self.entry_price_a = 0.0
        self.entry_price_b = 0.0
        self.entry_beta = 1.0 
        self.holding_time = 0 
        self.cooldown = 0 
        self.pnl_history = [0.0]
        self.cumulative_pnl = 0.0
        self.peak_pnl = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.current_pos = 0
        self.entry_price_a = 0.0
        self.entry_price_b = 0.0
        self.entry_beta = 1.0
        self.holding_time = 0
        self.cooldown = 0 
        self.pnl_history = [0.0]
        self.cumulative_pnl = 0.0
        self.peak_pnl = 0.0
        return self._get_obs(), {}

    def _get_obs(self):
        row = self.df.iloc[self.current_step]
        unrealized_pnl = 0.0
        if self.current_pos != 0 and self.entry_price_a > 0:
            c_pa, c_pb = row['price_a'], row['price_b']
            if not pd.isna(c_pa) and not pd.isna(c_pb):
                ret_a = (c_pa - self.entry_price_a) / max(self.entry_price_a, 1e-4)
                ret_b = (c_pb - self.entry_price_b) / max(self.entry_price_b, 1e-4)
                unrealized_pnl = self.current_pos * (ret_a - self.entry_beta * ret_b) / (1 + abs(self.entry_beta))
                
        vix_val = 0.0 if self.ablation_vix else (row['vix'] if not pd.isna(row['vix']) else 20.0)
        vix_norm = vix_val / 50.0
        zscore = row['z_score'] if not pd.isna(row['z_score']) else 0.0
        hl = row['half_life'] if not pd.isna(row['half_life']) else 15.0
        hl_norm = min(hl / 30.0, 5.0)
        
        hold_ratio = float(self.holding_time) / self.max_hold_days 
        cooldown_ratio = float(self.cooldown) / 5.0 
        
        spread_rsi = row['spread_rsi'] if not pd.isna(row['spread_rsi']) else 50.0
        norm_rsi = (spread_rsi - 50.0) / 50.0 
        
        return np.array([zscore, vix_norm, hl_norm, float(self.current_pos), unrealized_pnl, hold_ratio, cooldown_ratio, norm_rsi], dtype=np.float32)

    def step(self, action):
        target_pos = 0
        
        if self.cooldown > 0:
            self.cooldown -= 1
            target_pos = 0 
        else:
            if action == 1: target_pos = 1
            elif action == 2: target_pos = -1
            
            # 防止神經網路未收斂前產生當天強行多空互切狂刷手續費，強制先將翻單轉為平倉
            if (self.current_pos == 1 and target_pos == -1) or (self.current_pos == -1 and target_pos == 1):
                target_pos = 0

        row = self.df.iloc[self.current_step]
        c_pa, c_pb = row['price_a'], row['price_b']
        current_dynamic_beta = row.get('dynamic_beta', self.beta)
        
        is_delisted = pd.isna(c_pa) or pd.isna(c_pb) or c_pa <= 0.05 or c_pb <= 0.05
        if is_delisted: target_pos = 0
        
        time_stop_triggered = False
        if self.current_pos != 0:
            self.holding_time += 1
            if self.holding_time >= self.max_hold_days:
                target_pos = 0 
                time_stop_triggered = True
        else:
            self.holding_time = 0
        
        step_pnl = 0.0
        unrealized_pnl = 0.0
        forced_stop_loss = False
        trade_executed = 0

        if self.current_pos != 0 and self.current_step > 0:
            prev_row = self.df.iloc[self.current_step - 1]
            p_pa, p_pb = prev_row['price_a'], prev_row['price_b']
            
            if is_delisted:
                unrealized_pnl = self.stop_loss_pct * 2.0
                step_pnl = unrealized_pnl
            else:
                try:
                    ret_a = np.clip((c_pa - p_pa) / max(p_pa, 1e-4), -0.5, 0.5)
                    ret_b = np.clip((c_pb - p_pb) / max(p_pb, 1e-4), -0.5, 0.5)
                    step_pnl = self.current_pos * (ret_a - self.entry_beta * ret_b) / (1 + abs(self.entry_beta))
                    
                    unr_ret_a = (c_pa - self.entry_price_a) / max(self.entry_price_a, 1e-4)
                    unr_ret_b = (c_pb - self.entry_price_b) / max(self.entry_price_b, 1e-4)
                    unrealized_pnl = self.current_pos * (unr_ret_a - self.entry_beta * unr_ret_b) / (1 + abs(self.entry_beta))
                except:
                    step_pnl, unrealized_pnl = 0.0, 0.0
            
            if self.current_pos != 0 and (unrealized_pnl <= self.stop_loss_pct or is_delisted):
                forced_stop_loss = True
                target_pos = 0

        if (forced_stop_loss or time_stop_triggered) and self.current_pos != 0:
            self.cooldown = 5

        if target_pos != self.current_pos:
            trades_needed = abs(target_pos - self.current_pos)
            trade_penalty = trades_needed * self.tc
            step_pnl -= trade_penalty
            if target_pos != 0: trade_executed = 1
            
            if target_pos != 0 and not is_delisted:
                self.entry_price_a = c_pa
                self.entry_price_b = c_pb
                self.entry_beta = current_dynamic_beta 
                self.holding_time = 0
            else:
                self.entry_price_a, self.entry_price_b, unrealized_pnl = 0.0, 0.0, 0.0
                self.entry_beta = self.beta
                self.holding_time = 0

        # 極度厭惡過動機制：對每一次切換投資部位憑空施加大量懲罰，強制拉大進出間距
        action_changes = abs(target_pos - self.current_pos)
        reward_churn_penalty = action_changes * 1.5
        
        # 持股耐心獎勵：若沒有發生交易且原本處於持倉狀態，給予微小獎勵，鼓勵讓獲利奔跑
        holding_reward = 0.05 if (action_changes == 0 and self.current_pos != 0) else 0.0

        self.current_pos = target_pos
        self.pnl_history.append(step_pnl)
        self.cumulative_pnl += step_pnl
        
        if self.cumulative_pnl > self.peak_pnl: self.peak_pnl = self.cumulative_pnl
        current_drawdown = (self.peak_pnl - self.cumulative_pnl) / self.peak_pnl if self.peak_pnl > 0 else 0

        reward = step_pnl * 100.0

        if current_drawdown > 0.02: reward -= (current_drawdown ** 2) * 50.0
        if forced_stop_loss: reward -= 0.5
        if time_stop_triggered: reward -= 0.1 
        reward -= reward_churn_penalty
        reward += holding_reward

        self.current_step += 1
        terminated = bool(self.current_step >= self.max_steps)
        info = {'step_pnl': step_pnl, 'trade_executed': trade_executed}

        return self._get_obs(), float(np.clip(reward, -10.0, 10.0)), terminated, False, info

In [6]:
# ==========================================
# 模組 5: 滾動式推進分析與資產管理 (WFO Engine)
# ==========================================
class RLPairsTradingWFO:
    def __init__(self, price_pivot, vix_features, sector_map):
        self.price_pivot = price_pivot
        self.vix_features = vix_features
        self.sector_map = sector_map
        self.all_dates = price_pivot.index
        self.plots_generated = 0 
        
    def prepare_rl_features(self, df, beta, z_window=Z_WINDOW, rsi_window=RSI_WINDOW):
        df = df.copy()
        
        roll_cov = df['price_a'].rolling(window=z_window).cov(df['price_b'])
        roll_var = df['price_b'].rolling(window=z_window).var()
        df['dynamic_beta'] = (roll_cov / roll_var).fillna(beta)
        
        df['spread'] = df['price_a'] - df['dynamic_beta'] * df['price_b']
        
        roll_mean = df['spread'].ewm(span=z_window, adjust=False).mean()
        roll_std = df['spread'].ewm(span=z_window, adjust=False).std()
        df['z_score'] = (df['spread'] - roll_mean) / roll_std
        
        delta = df['spread'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=rsi_window).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=rsi_window).mean()
        rs = gain / loss.replace(0, 1e-6) # 避免除以零
        df['spread_rsi'] = 100 - (100 / (1 + rs))
        df['spread_rsi'] = df['spread_rsi'].fillna(50.0)
        
        hl_list = []
        last_valid_hl = 15.0
        for i in range(len(df)):
            if i < z_window: hl_list.append(last_valid_hl)
            elif i % 20 == 0:
                window_ts = df['spread'].iloc[i-z_window:i].dropna()
                current_hl = PairSelector.compute_half_life(window_ts.values)
                hl_list.append(current_hl)
                last_valid_hl = current_hl
            else:
                hl_list.append(hl_list[-1])
        df['half_life'] = hl_list
        return df.ffill().fillna(0)

    def plot_pair_behavior(self, stk_a, stk_b, test_df, nav_records, beta, pos_records):
        fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                            subplot_titles=(
                                f'【{stk_a} vs {stk_b}】標準化價格走勢 (Initial Beta: {beta:.2f})', 
                                '價差 Z-Score 走勢與 AI 進出場點', 
                                'RL Agent 獨立淨值走勢 ($)'
                            ))
        
        norm_a = test_df['price_a'] / test_df['price_a'].iloc[0]
        norm_b = test_df['price_b'] / test_df['price_b'].iloc[0]
        dates = test_df['date'].tolist()
        z_scores = test_df['z_score'].tolist()
        
        fig.add_trace(go.Scatter(x=dates, y=norm_a, name=f'{stk_a} (Norm)', line=dict(color='blue')), row=1, col=1)
        fig.add_trace(go.Scatter(x=dates, y=norm_b, name=f'{stk_b} (Norm)', line=dict(color='orange')), row=1, col=1)
        
        fig.add_trace(go.Scatter(x=dates, y=z_scores, name='Z-Score', line=dict(color='purple')), row=2, col=1)
        fig.add_trace(go.Scatter(x=dates, y=[0]*len(dates), name='Mean (0)', line=dict(color='black', dash='dash')), row=2, col=1)
        fig.add_trace(go.Scatter(x=dates, y=[2]*len(dates), name='+2 Std', line=dict(color='red', dash='dot')), row=2, col=1)
        fig.add_trace(go.Scatter(x=dates, y=[-2]*len(dates), name='-2 Std', line=dict(color='green', dash='dot')), row=2, col=1)
        
        long_dates, long_z = [], []
        short_dates, short_z = [], []
        exit_dates, exit_z = [], []
        
        for i in range(1, len(pos_records)):
            prev_pos = pos_records[i-1]
            curr_pos = pos_records[i]
            
            if curr_pos != prev_pos:
                if curr_pos == 1:
                    long_dates.append(dates[i])
                    long_z.append(z_scores[i])
                elif curr_pos == -1:
                    short_dates.append(dates[i])
                    short_z.append(z_scores[i])
                elif curr_pos == 0:
                    exit_dates.append(dates[i])
                    exit_z.append(z_scores[i])

        if long_dates:
            fig.add_trace(go.Scatter(x=long_dates, y=long_z, mode='markers', name='做多價差 (Long Spread)',
                                     marker=dict(symbol='triangle-up', size=12, color='green', line=dict(width=1, color='darkgreen'))), row=2, col=1)
        if short_dates:
            fig.add_trace(go.Scatter(x=short_dates, y=short_z, mode='markers', name='做空價差 (Short Spread)',
                                     marker=dict(symbol='triangle-down', size=12, color='red', line=dict(width=1, color='darkred'))), row=2, col=1)
        if exit_dates:
            fig.add_trace(go.Scatter(x=exit_dates, y=exit_z, mode='markers', name='平倉 (Flat)',
                                     marker=dict(symbol='x', size=10, color='black', line=dict(width=2, color='black'))), row=2, col=1)

        fig.add_trace(go.Scatter(x=dates, y=nav_records, name='Agent NAV', line=dict(color='crimson')), row=3, col=1)
        
        fig.update_layout(height=800, title_text=f"RL 配對交易觀測站: {stk_a} & {stk_b}", template='plotly_white')
        fig.show()

    def run_wfo(self, plot_sample_pairs=False, max_plots=3, target_plot_pairs=None):
        print(f"\n啟動多視窗整合回測引擎 (Daily Portfolio Manager)")
        print(f"總資金: ${INITIAL_CAPITAL} | 單注: ${TRANCHE_ALLOCATION} | 梯隊滾動: {ROLLING_WINDOW} 天")
        
        cash = INITIAL_CAPITAL
        active_tranches = []
        completed_tranches_count = 0
        historical_dates = []
        historical_portfolio_nav = []
        total_trades_count = 0
        failed_due_to_margin = False

        for t in range(FORMATION_WINDOW, len(self.all_dates)):
            current_date = self.all_dates[t]
            
            still_active = []
            for tr in active_tranches:
                if current_date >= tr['end_date']:
                    final_nav = tr['series'].iloc[-1]
                    cash += final_nav
                    completed_tranches_count += 1
                else:
                    still_active.append(tr)
            active_tranches = still_active

            daily_total_nav = cash
            for tr in active_tranches:
                latest_val = tr['series'].loc[:current_date]
                if not latest_val.empty:
                    daily_total_nav += latest_val.iloc[-1]

            if daily_total_nav <= 1000.0:
                print(f"破產警報！系統總資產(${daily_total_nav:,.2f})已低於 $1000 最低維運標準，交易強制永久終止。")
                failed_due_to_margin = True
                break

            if (t - FORMATION_WINDOW) % ROLLING_WINDOW == 0:
                if cash >= TRANCHE_ALLOCATION:
                    cash -= TRANCHE_ALLOCATION
                    train_start_date = self.all_dates[t - FORMATION_WINDOW]
                    train_end_date = self.all_dates[t]
                    test_end_idx = min(t + TRADING_WINDOW, len(self.all_dates) - 1)
                    test_end_date = self.all_dates[test_end_idx]

                    print(f"\n--- 新梯隊啟動 | 系統剩餘現金: ${cash:.2f} | 執行區間: {train_end_date.date()} ~ {test_end_date.date()} ---")
                    
                    train_pivot = self.price_pivot.loc[train_start_date:train_end_date]
                    train_vix = self.vix_features.loc[train_start_date:train_end_date]['VIX']
                    
                    train_features_std, _ = PairSelector.feature_engineering_pipeline(train_pivot, train_vix)
                    top_pairs_dict = PairSelector.select_pairs_with_hdbscan(train_pivot, train_features_std, self.sector_map, [MAX_PAIRS_PER_TRANCHE])
                    actual_pairs = top_pairs_dict.get(MAX_PAIRS_PER_TRANCHE, [])

                    if not actual_pairs:
                        print("▲ 無高品質共整配對，放棄建倉，資金輪空不使用。")
                        cash += TRANCHE_ALLOCATION
                    else:
                        sub_allocation = TRANCHE_ALLOCATION / len(actual_pairs)
                        
                        tranche_nav_records = None
                        tranche_dates = []
                        window_pair_results = [] 

                        for p_idx, pair in enumerate(actual_pairs):
                            stk_a, stk_b, beta_coef = pair['stock_a'], pair['stock_b'], pair['beta']
                            sector = pair.get('sector', 'Unknown')
                            ssd = pair.get('ssd', 0.0)
                            print(f"選定配對: {stk_a:<5} & {stk_b:<5} | Sector: {sector:<25} | SSD: {ssd:>6.4f} | Beta: {beta_coef:.4f}")
                            
                            rl_train_raw = pd.DataFrame({'price_a': train_pivot[stk_a], 'price_b': train_pivot[stk_b], 'vix': train_vix})
                            rl_train_df = self.prepare_rl_features(rl_train_raw, beta_coef)
                            
                            env_train = PairsTradingEnv(rl_train_df, TRANSACTION_COST, stop_loss_pct=STOP_LOSS_PCT, max_hold_days=MAX_HOLD_DAYS, beta=beta_coef)
                            vec_env_train = DummyVecEnv([lambda: env_train])
                            
                            # 【核心修改】讓 PPO 使用全域的學習參數，方便日後在上方一次性調整
                            model = PPO('MlpPolicy', vec_env_train, verbose=0, 
                                        learning_rate=RL_LEARNING_RATE, 
                                        gamma=RL_GAMMA,
                                        ent_coef=RL_ENT_COEF,
                                        batch_size=RL_BATCH_SIZE)
                            
                            model.learn(total_timesteps=RL_TOTAL_TIMESTEPS)
                            
                            ext_test_start = train_end_date - pd.Timedelta(days=120)
                            test_pivot = self.price_pivot.loc[ext_test_start:test_end_date]
                            test_vix = self.vix_features.loc[ext_test_start:test_end_date]['VIX']
                            
                            rl_test_raw = pd.DataFrame({'price_a': test_pivot.get(stk_a, pd.Series(dtype=float)), 'price_b': test_pivot.get(stk_b, pd.Series(dtype=float)), 'vix': test_vix})
                            rl_test_full = self.prepare_rl_features(rl_test_raw, beta_coef)
                            actual_test_df = rl_test_full[rl_test_full.index >= train_end_date].reset_index()
                            
                            if len(actual_test_df) >= 10:
                                env_test = PairsTradingEnv(actual_test_df, TRANSACTION_COST, stop_loss_pct=STOP_LOSS_PCT, max_hold_days=MAX_HOLD_DAYS, beta=beta_coef)
                                vec_env_test = DummyVecEnv([lambda: env_test])
                                obs = vec_env_test.reset()
                                dones = [False]
                                
                                sub_nav = sub_allocation
                                sub_records = [sub_nav]
                                pos_records = [0] 
                                tranche_dates = actual_test_df['date'].tolist()
                                
                                while not dones[0]:
                                    action, _ = model.predict(obs, deterministic=True)
                                    obs, _, dones, infos = vec_env_test.step(action)
                                    
                                    current_pos = obs[0][3]
                                    pos_records.append(current_pos)
                                    
                                    total_trades_count += infos[0].get('trade_executed', 0)
                                    sub_nav += infos[0].get('step_pnl', 0.0) * sub_allocation 
                                    sub_records.append(sub_nav)
                                
                                pair_name = f"{stk_a}-{stk_b}"
                                pair_final_nav = sub_records[-1]
                                pair_net_pnl = pair_final_nav - sub_allocation
                                pair_ret = (pair_net_pnl / sub_allocation) * 100
                                window_pair_results.append({
                                    'name': pair_name,
                                    'allocation': sub_allocation,
                                    'pnl': pair_net_pnl,
                                    'ret': pair_ret
                                })

                                should_plot = False
                                if target_plot_pairs is not None:
                                    if pair_name in target_plot_pairs:
                                        should_plot = True
                                elif plot_sample_pairs and self.plots_generated < max_plots:
                                    should_plot = True

                                if should_plot:
                                    self.plot_pair_behavior(stk_a, stk_b, actual_test_df, sub_records, beta_coef, pos_records)
                                    self.plots_generated += 1

                                if tranche_nav_records is None:
                                    tranche_nav_records = np.array(sub_records)
                                else:
                                    tranche_nav_records += np.array(sub_records)

                        if window_pair_results:
                            print("\n---視窗內各配對損益結算---")
                            window_total_pnl = 0
                            window_capital = len(actual_pairs) * sub_allocation
                            for res in window_pair_results:
                                window_total_pnl += res['pnl']
                                print(f"配對 {res['name']:<12} | 分配資金: ${res['allocation']:>3.0f} | 淨損益: ${res['pnl']:>8.2f} | 區間報酬: {res['ret']:>7.2f}%")
                            
                            window_roi = (window_total_pnl / window_capital) * 100 if window_capital > 0 else 0.0
                            print(f">>> 【視窗結算】 視窗投入本金:${window_capital:,.0f} | 視窗總淨利:${window_total_pnl:,.2f} | 總報酬:{window_roi:.2f}%")

                        if tranche_nav_records is not None:
                            daily_series = pd.Series(tranche_nav_records, index=tranche_dates)
                            active_tranches.append({'end_date': tranche_dates[-1], 'series': daily_series})
                else:
                    print(f"▲ 現金水位(${cash:,.2f})暫時不足以提撥 10% 資金(${TRANCHE_ALLOCATION:,.2f})，新梯隊輪空！")

            historical_dates.append(current_date)
            historical_portfolio_nav.append(daily_total_nav)

        self.results_df = pd.DataFrame({'Date': historical_dates, 'Agent_NAV': historical_portfolio_nav}).set_index('Date')
        self.completed_tranches_count = completed_tranches_count
        self.total_trades_count = total_trades_count
        self.failed = failed_due_to_margin

    def print_and_plot(self):
        print("\n回測迴圈結束！")
        if self.failed: return

        results_df = self.results_df
        total_return = (results_df['Agent_NAV'].iloc[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL
        results_df['Daily_Return'] = results_df['Agent_NAV'].pct_change().fillna(0)
        years = len(results_df) / 252.0
        
        annualized_return = ((results_df['Agent_NAV'].iloc[-1] / INITIAL_CAPITAL) ** (1/years) - 1) if years > 0 else 0.0
        annualized_volatility = results_df['Daily_Return'].std() * (252 ** 0.5)
        sharpe_ratio = (annualized_return / annualized_volatility) if annualized_volatility > 0 else 0.0
        
        results_df['Agent_Peak'] = results_df['Agent_NAV'].cummax()
        results_df['Drawdown'] = (results_df['Agent_NAV'] - results_df['Agent_Peak']) / results_df['Agent_Peak']
        mdd = results_df['Drawdown'].min()
        calmar_ratio = (annualized_return / abs(mdd)) if mdd < 0 else 0.0

        print("\n【滾動梯隊資金管理 - 最終總結結算】")
        print(f"最終資金餘額: ${results_df['Agent_NAV'].iloc[-1]:,.2f}")
        print(f"累計淨報酬率: {total_return * 100:.2f}%")
        print(f"年化報酬率(CAGR): {annualized_return * 100:.2f}%")
        print(f"年化波動率(Volatility): {annualized_volatility * 100:.2f}%")
        print(f"最大回撤(MDD): {mdd * 100:.2f}%")
        print(f"年化夏普比率(Sharpe Ratio): {sharpe_ratio:.2f}")
        print(f"卡瑪比率(Calmar Ratio): {calmar_ratio:.2f}")
        print(f"成功執行迴歸的梯隊: {self.completed_tranches_count} 梯")
        print(f"全期間總交易次數: {self.total_trades_count} 次\n")

        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, row_heights=[0.7, 0.3],
                            subplot_titles=('Rolling Tranches Portfolio NAV', 'Strategy Drawdown Analysis (%)'))
        
        fig.add_trace(go.Scatter(x=results_df.index, y=results_df['Agent_NAV'], name='Portfolio NAV ($)', line=dict(color='indigo')), row=1, col=1)
        fig.add_trace(go.Scatter(x=results_df.index, y=results_df['Drawdown'] * 100, name='Drawdown (%)', fill='tozeroy', line=dict(color='crimson')), row=2, col=1)
        
        fig.update_layout(title=f'Rolling Tranches Strategy (Initial: ${INITIAL_CAPITAL})', height=800, template='plotly_white')
        fig.show()

In [7]:
# ==========================================
# 執行區域 (Main Block)
# ==========================================
if __name__ == "__main__":
    start_wall_time = time.time()
    start_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"訓練開始時間: {start_timestamp}")

    data_fetcher = FetchDataRL(
        db_path=DB_PATH,
        start_date=START_DATE,
        end_date=END_DATE,
        target_sector=TARGET_SECTOR,
        use_dynamic=USE_DYNAMIC_SECTORS,
        min_days=MIN_HISTORY_DAYS,
        imputed_path=IMPUTED_SECTOR_PATH
    )
    prices_raw, price_pivot, vix_features, sector_map = data_fetcher.fetch_and_preprocess()

    wfo_engine = RLPairsTradingWFO(price_pivot, vix_features, sector_map)
    
    wfo_engine.run_wfo(plot_sample_pairs=True, max_plots=50, target_plot_pairs=None)
    
    wfo_engine.print_and_plot()

    end_wall_time = time.time()
    total_seconds = end_wall_time - start_wall_time
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = int(total_seconds % 60)
    
    print(f"\n訓練結束時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"總共執行耗時: {hours} 小時 {minutes} 分 {seconds} 秒")

訓練開始時間: 2026-04-18 10:29:36
從資料庫載入原始資料...
目前嘗試連接的資料庫絕對路徑為: c:\Clark\YZU\Papper\Code\data\sp500.db
成功加載價格數據: 616,467 筆
成功加載行業數據: 843 筆
從本機快取載入已補齊的產業分類: ..\data\imputed_sectors.csv
全市場模式: 使用全部 S&P 500 股票
Matrix: 1008 days x 623 tickers
Fetching VIX data...
VIX feature matrix aligned.

最終股票池統計:
  股票數量: 623
  時間範圍: 2019-01-01 至 2022-12-31

啟動多視窗整合回測引擎 (Daily Portfolio Manager)
總資金: $10000 | 單注: $1428.5714285714287 | 梯隊滾動: 21 天

--- 新梯隊啟動 | 系統剩餘現金: $8571.43 | 執行區間: 2020-01-02 ~ 2020-07-02 ---
啟動特徵工程 Pipeline...
啟動多執行緒處理 623 檔股票之時間序列與 ADF 微觀特徵...
特徵矩陣建置完成！維度: (257, 10)
完成排列組合，共產生 701 組配對
P-value 檢定過濾後，剩餘 4 組配對。
  -> 經過獲利空間與過零率校驗後，最終剩餘 3 組配對可以進行評分。
SSD 排序完成，前部配對:
  1: BG vs WMT (SSD: 49.2539, Beta: 0.5096)
  2: CBOE vs ICE (SSD: 21.5221, Beta: 1.2489)
  3: BG vs KO (SSD: 60.1554, Beta: 0.5015)
選定配對: BG    & WMT   | Sector: Consumer Staples          | SSD: 49.2539 | Beta: 0.5096


選定配對: CBOE  & ICE   | Sector: Financials                | SSD: 21.5221 | Beta: 1.2489


選定配對: BG    & KO    | Sector: Consumer Staples          | SSD: 60.1554 | Beta: 0.5015



---視窗內各配對損益結算---
配對 BG-WMT       | 分配資金: $476 | 淨損益: $  -82.83 | 區間報酬:  -17.39%
配對 CBOE-ICE     | 分配資金: $476 | 淨損益: $  -12.94 | 區間報酬:   -2.72%
配對 BG-KO        | 分配資金: $476 | 淨損益: $   94.22 | 區間報酬:   19.79%
>>> 【視窗結算】 視窗投入本金:$1,429 | 視窗總淨利:$-1.55 | 總報酬:-0.11%

--- 新梯隊啟動 | 系統剩餘現金: $7142.86 | 執行區間: 2020-02-03 ~ 2020-08-03 ---
啟動特徵工程 Pipeline...
啟動多執行緒處理 623 檔股票之時間序列與 ADF 微觀特徵...
特徵矩陣建置完成！維度: (280, 10)
完成排列組合，共產生 827 組配對
P-value 檢定過濾後，剩餘 4 組配對。
  -> 經過獲利空間與過零率校驗後，最終剩餘 4 組配對可以進行評分。
SSD 排序完成，前部配對:
  1: MCK vs STE (SSD: 3.7989, Beta: 0.7775)
  2: AMAT vs TER (SSD: 10.3314, Beta: 0.7240)
  3: CDW vs TER (SSD: 122.7788, Beta: 1.4148)
  4: MKC vs TSN (SSD: 54.7180, Beta: 0.6259)
選定配對: MCK   & STE   | Sector: Health Care               | SSD: 3.7989 | Beta: 0.7775


選定配對: AMAT  & TER   | Sector: Information Technology    | SSD: 10.3314 | Beta: 0.7240


...

選定配對: ETN   & HUBB  | Sector: Industrials               | SSD: 28.1708 | Beta: 0.8467


選定配對: EQT   & WMB   | Sector: Energy                    | SSD: 3.4867 | Beta: 0.9850


選定配對: AVB   & FRT   | Sector: Real Estate               | SSD: 42.4392 | Beta: 1.2531


選定配對: EQT   & RRC   | Sector: Energy                    | SSD: 65.4906 | Beta: 1.1899

---視窗內各配對損益結算---
配對 A-ZBH        | 分配資金: $286 | 淨損益: $  -35.51 | 區間報酬:  -12.43%
配對 ETN-HUBB     | 分配資金: $286 | 淨損益: $  -13.14 | 區間報酬:   -4.60%
配對 EQT-WMB      | 分配資金: $286 | 淨損益: $  -30.71 | 區間報酬:  -10.75%
配對 AVB-FRT      | 分配資金: $286 | 淨損益: $  -20.63 | 區間報酬:   -7.22%
配對 EQT-RRC      | 分配資金: $286 | 淨損益: $  -48.23 | 區間報酬:  -16.88%
>>> 【視窗結算】 視窗投入本金:$1,429 | 視窗總淨利:$-148.23 | 總報酬:-10.38%

--- 新梯隊啟動 | 系統剩餘現金: $1152.05 | 執行區間: 2021-06-03 ~ 2021-12-01 ---
啟動特徵工程 Pipeline...
啟動多執行緒處理 623 檔股票之時間序列與 ADF 微觀特徵...
特徵矩陣建置完成！維度: (435, 10)
完成排列組合，共產生 1020 組配對
P-value 檢定過濾後，剩餘 22 組配對。
  -> 經過獲利空間與過零率校驗後，最終剩餘 21 組配對可以進行評分。
SSD 排序完成，前部配對:
  1: DECK vs HAS (SSD: 4.4969, Beta: 1.1214)
  2: AMZN vs PHM (SSD: 5.3316, Beta: 0.9786)
  3: MAT vs UA (SSD: 1.7823, Beta: 1.0123)
  4: ENPH vs IPGP (SSD: 125.3247, Beta: 1.6539)
  5: DRI vs HLT (SSD: 6.3121, Beta: 1.2110)
選定配對: DECK  & HAS   | Sector: Consumer Discretionary    | S


訓練結束時間: 2026-04-18 14:21:27
總共執行耗時: 3 小時 51 分 50 秒
